In [1]:
%cd ../../..

/home/hoanghu/projects/Food-Waste-Optimization


In [2]:
from pathlib import Path

import pandas as pd
import psycopg as pg
from loguru import logger
from psycopg import sql
from psycopg.rows import dict_row

In [ ]:
USER = ""
PWD = ""
PORT = ""
HOST = ""
DB_NAME = ""

# Upload table `menus`

In [4]:
path_dir = Path("data/processed/phase_4/menus")

list_df = [pd.read_parquet(path) for path in path_dir.glob("*.parquet")]
df = pd.concat(list_df)

df.head()

,date,restaurant,meal_ids,fitness
0,2025-01-14,phy,"[6353, 950004, 950017]",1.666848
1,2025-01-15,phy,"[6137, 950004, 2207]",2.503424
2,2025-01-16,phy,"[9105, 3012, 1516]",1.615522
3,2025-01-15,phy,"[6356, 6137, 6552]",2.675377
4,2025-01-17,phy,"[6138, 950004, 6140]",3.129494


In [5]:
values = []

for r in df.itertuples():
    row = (r.date, r.restaurant, r.meal_ids.tolist(), r.fitness)

    values.append(row)

In [6]:
try:
    with pg.connect(
        user=USER,
        password=PWD,
        host=HOST,
        port=PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:

            # Compose SQL
            cols = ['date', 'restaurant', 'meal_ids', 'fitness']
            table = 'menu'

            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            # for v in values:
            cur.executemany(stmt, values)
            conn.commit()

            # ret = cur.fetchall()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")

# Upload table `meals`

In [7]:
path = "data/processed/phase_4/dim_meals.xlsx"
dim_meals_raw = pd.read_excel(path)
dim_meals_raw.head()

,meal_id,meal_type_1,schoolyear,is_kela,is_new,restaurant,meal_type_2,pcs_mean
0,9017,vegan,24-25,True,True,che-exa,vegan-miscellaneous,140.165854
1,7201,vegan,23-24,False,False,NaN,NaN,121.918212
2,9032,vegan,23-24,False,False,NaN,NaN,121.918212
3,9102,vegan,23-24,False,False,NaN,NaN,121.918212
4,7010,vegetarian,24-25,False,False,che-exa,NaN,71.000000


In [8]:
def _conv_restaurant(s: str):
    match (s):
        case 'che-exa':
            out = ['che', 'exa']
        case 'phy':
            out = ['phy']
        case _:
            out = []

    return out

values = [
    (r.meal_id, r.meal_type_1, r.schoolyear, r.is_kela, r.is_new, _conv_restaurant(r.restaurant), r.meal_type_2, r.pcs_mean)
    for r in dim_meals_raw.itertuples()
]

In [9]:
# Compose SQL
cols = dim_meals_raw.columns
table = 'meals'

try:
    with pg.connect(
        user=USER,
        password=PWD,
        host=HOST,
        port=PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:
            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            # for v in values:
            cur.executemany(stmt, values)
            conn.commit()

            # ret = cur.fetchall()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")

# Upload table `meal_names`

In [10]:
path = "data/processed/phase_4/dim_meal_names.xlsx"
dim_meal_names_raw = pd.read_excel(path)
dim_meal_names_raw.head()

,meal_id,meal
0,9017,"""Butter"" härkäpapua & pähkinää"
1,7201,2023 Härkäpu-sienilasagnette
2,9032,Appelisiini-luomukikhernecurrya
3,9102,Artisokkavugetteja & tuoretomaattisalsaa
4,7010,Aurajuusto-pinaattilasagnette


In [11]:
values = [
    (r.meal_id, r.meal)
    for r in dim_meal_names_raw.itertuples()
]

In [12]:
# Compose SQL
cols = dim_meal_names_raw.columns
table = 'meal_names'

try:
    with pg.connect(
        user=USER,
        password=PWD,
        host=HOST,
        port=PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:
            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            # for v in values:
            cur.executemany(stmt, values)
            conn.commit()

            # ret = cur.fetchall()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")

# Upload table `biowaste`

In [13]:
path = "data/processed/phase_4/dim_waste.xlsx"
dim_waste_raw = pd.read_excel(path)
dim_waste_raw.head()

,meal_id,waste
0,9017,0.010000
1,7201,0.010000
2,9032,0.010000
3,9102,0.010000
4,7010,0.067843


In [14]:
values = [
    (r.meal_id, r.waste)
    for r in dim_waste_raw.itertuples()
]

In [15]:
# Compose SQL
cols = dim_waste_raw.columns
table = 'biowaste'

try:
    with pg.connect(
        user=USER,
        password=PWD,
        host=HOST,
        port=PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:
            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            # for v in values:
            cur.executemany(stmt, values)
            conn.commit()

            # ret = cur.fetchall()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")

# Upload table `co2`

In [16]:
path = "data/processed/phase_4/dim_co2.xlsx"
dim_co2_raw = pd.read_excel(path)
dim_co2_raw.head()

,meal_id,co2
0,34,0.81
1,37,0.61
2,710,0.67
3,713,0.56
4,724,0.82


In [17]:
values = [
    (r.meal_id, r.co2)
    for r in dim_co2_raw.itertuples()
]

In [18]:
# Compose SQL
cols = dim_co2_raw.columns
table = 'co2'

try:
    with pg.connect(
        user=USER,
        password=PWD,
        host=HOST,
        port=PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:
            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            cur.executemany(stmt, values)
            conn.commit()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")

# Upload table `pieces_whole`

In [19]:
path = "data/processed/phase_4/dim_pieces_whole.xlsx"
dim_pieces_whole_raw = pd.read_excel(path)
dim_pieces_whole_raw.head()

,date,pcs,restaurant
0,2024-09-02,759.74,che
1,2024-09-03,708.30,che
2,2024-09-04,664.79,che
3,2024-09-05,666.73,che
4,2024-09-06,732.02,che


In [20]:
values = [
    (r.date, r.pcs, r.restaurant)
    for r in dim_pieces_whole_raw.itertuples()
]

In [21]:
# Compose SQL
cols = dim_pieces_whole_raw.columns
table = 'pieces_whole'

try:
    with pg.connect(
        user=USER,
        password=PWD,
        host=HOST,
        port=PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:
            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            cur.executemany(stmt, values)
            conn.commit()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")